# 00: Multi-Unit Raster Mosaicking (GDAL VRT)

This notebook demonstrates how to use the `landseg` raster mosaicking utility (`build_vrt_mosaic`) to assemble disparate raster tiles into a unified, reprojected **GDAL Virtual Raster (`.vrt`)** mosaic.

### Context & Motivation
In regional and provincial remote sensing workflows (e.g. Ontario Crown Forest Management Units), imagery and thematic landcover layers often span multiple UTM zones (e.g. UTM Zone 16N, 17N, 18N) or administrative unit boundaries. 

Rather than reserializing gigabytes of heavy imagery to disk, `build_vrt_mosaic`:
1. **Zero Pixel Duplication:** Assembles tiles purely in virtual XML metadata.
2. **Cross-Projection Reprojection:** Transparently warps tiles from differing source projections (e.g., UTM 16N/17N/18N) into the canonical Ontario provincial grid (`EPSG:3161` Ontario MNR Lambert).
3. **Categorical & Continuous Modes:** Preserves discrete integer class boundaries (via nearest-neighbor) or resamples continuous imagery (bilinear, cubic, lanczos).
4. **Direct Pipeline Integration:** The resulting `.vrt` file can be passed directly as an input layer into `dataset_manifest.json` for subsequent harmonization, tiling, and model training.

## Environment Setup & Imports

In [ ]:
# If you are running this notebook on **Databricks**
# uncomment and run the following line in the cell below
# to install the local `landseg` package in your cluster's Python environment:

# %pip install -e ..

In [ ]:
# standard imports
import os
import pathlib
import subprocess
import sys
# third-party imports
import rasterio
# local imports
sys.path.insert(0, os.path.join(os.path.abspath('..'), 'src'))
import landseg.geopipe.harmonize as harmonize

print('Successfully imported landseg and dependencies!')

# Check if running in a Databricks environment
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

# Define the experiment root path where artifacts and results will be stored.
# Relative paths are standard for local repositories. On Databricks, consider utilizing
# absolute paths on a Unity Catalog Volume or DBFS path (e.g. '/Volumes/my_catalog/my_schema/my_volume/experiment/').
EXP_ROOT = '../experiment/'

if IS_DATABRICKS:
    # EXP_ROOT = '/Volumes/my_catalog/my_schema/my_volume/experiment/'
    print("Running on Databricks. For high performance, configure EXP_ROOT using a Unity Catalog Volume or DBFS path.")

## Generate Dummy Dataset

For testing purposes we can use the shipped utility to generate a set of real rasters (`.tif` files) to demo the following process

In [ ]:
subprocess.run(
    [
        sys.executable,
        '../scripts/generate_dummy_data.py',
        '--output_dir',
        f'../{EXP_ROOT}/input',
        '-y'    # -y flag will overwrite existing data files
    ],
    check=True
)

## Collect I/O for Raster Mosaicking

If dummay dataset is generated, we collect dummy data file paths for inputs (see `landseg/testing/pre_mosaic.py` for details). For real-data runs you should replace with actualy file paths

Note:
- `images_to_mosaic` and `labels_to_mosaic` require a list of file paths
- `image_mosaic` and `label_mosaic` require a regular file path
- both `str` and `pathlib.Path` types are supported

In [ ]:
root = pathlib.Path(EXP_ROOT)

# image mosaicking I/O
images_to_mosaic = [
    root / 'input' / 'utm_rasters' / 'image_17n.tif',
    root / 'input' / 'utm_rasters' / 'image_18n.tif',
]
image_mosaic = root / 'input' / 'utm_rasters' / 'image_mosaic.vrt'

# label mosaicking I/O
labels_to_mosaic = [
    root / 'input' / 'utm_rasters' / 'label_17n.tif',
    root / 'input' / 'utm_rasters' / 'label_18n.tif',
]
label_mosaic = root / 'input' / 'utm_rasters' / 'label_mosaic.vrt'

## Build Continuous Imagery VRT Mosaic

We now invoke `harmonize.build_vrt_mosaic()` to merge both tiles into a single virtual raster reprojected to Ontario MNR Lambert (`EPSG:3161`).

Key options used:
- `target_crs='EPSG:3161'`
- `resampling_method='bilinear'` for image data
- `resampling_method='nearest'` for label data
- `target_resolution=20.0` (20-meter pixel resolution)

In [ ]:
# ----- mosaick images and inspect
img_mosaic_path = harmonize.build_vrt_mosaic(
    input_paths=images_to_mosaic,
    output_path=image_mosaic,
    target_crs='EPSG:3161',
    target_resolution=20.0,
    resampling_method='bilinear',
    is_categorical=False,
    relative_paths=True,
)
print(f'Image Mosaic Created at: {img_mosaic_path}')

with rasterio.open(img_mosaic_path) as ds:
    print(f'CRS: {ds.crs}')
    print(f'Dimensions (W x H): {ds.width} x {ds.height}')
    print(f'Band Count: {ds.count}')
    print(f'Band Descriptions: {ds.descriptions}')
    print(f'Bounds: {ds.bounds}')
    print(f'Pixel Resolution: ({ds.res[0]:.2f}m, {ds.res[1]:.2f}m)')

# ----- mosaick labels and inspect
lbl_mosaic_path = harmonize.build_vrt_mosaic(
    input_paths=labels_to_mosaic,
    output_path=label_mosaic,
    target_crs='EPSG:3161',
    target_resolution=20.0,
    resampling_method='nearest',
    is_categorical=True,
    relative_paths=True,
)
print(f'Label Mosaic Created at: {lbl_mosaic_path}')

with rasterio.open(lbl_mosaic_path) as ds:
    print(f'CRS: {ds.crs}')
    print(f'Dimensions (W x H): {ds.width} x {ds.height}')
    print(f'Band Count: {ds.count}')
    print(f'Band Descriptions: {ds.descriptions}')
    print(f'Bounds: {ds.bounds}')
    print(f'Pixel Resolution: ({ds.res[0]:.2f}m, {ds.res[1]:.2f}m)')

## CLI Equivalent Usage

You can also run this exact workflow from the command line using the dedicated script under `scripts/`:

```bash
python scripts/build_raster_mosaic.py \
    /dir/to/unit/rasters/*.tif \
    --output experiment/demo_mosaic/mosaic_cli.vrt \
    --target-crs EPSG:3161 \
    --resampling bilinear \
    --relative-paths
```

### Next Steps
Now that your regional tiles are assembled into unified VRT layers, proceed to **`01_data_preparation.ipynb`** to define your canonical World Grid, compile the dataset manifest, and run data harmonization and ingestion.